# Function vectors: lift a task out of in-context examples and add it zero-shot

Given a few `Q: France / A: Paris` examples, a language model works out the task and
answers the next query. Function vectors locate where that happens: a small set of
attention heads whose last-token outputs carry the task. Their mean output, summed, is the
function vector. Added to the residual stream of a prompt with no examples at all, it makes
the model perform the task.

Paper: Eric Todd, Millicent L. Li, Arnab Sen Sharma, Aaron Mueller, Byron C. Wallace and
David Bau, "Function Vectors in Large Language Models", ICLR 2024,
https://arxiv.org/abs/2310.15213.

The steps follow the paper on GPT-2 small (124M parameters, CPU) and a country-to-capital
task: (1) per-head outputs at the last token from `torchlens.mechinterp.attention_head_contributions`;
(2) each head's causal indirect effect: put its mean clean output into prompts whose example
answers are shuffled, and measure how much the correct answer's probability recovers;
(3) sum the top heads' mean outputs into the function vector and add it with `tl.steer`
at the last position of a zero-shot prompt. The first cell downloads GPT-2 small from the
Hugging Face Hub at a pinned revision.

In [ ]:
# recipe-test: tool=transformers expected=True
import copy
import random

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

import torchlens as tl
import torchlens.mechinterp as mi

REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"  # pragma: allowlist secret
model = GPT2LMHeadModel.from_pretrained(
    "openai-community/gpt2", revision=REVISION, attn_implementation="eager"
).eval()
model.config.use_cache = False
model.requires_grad_(False)
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2", revision=REVISION)

CAPITALS = """France Paris|Germany Berlin|Italy Rome|Spain Madrid|Japan Tokyo|China Beijing
Russia Moscow|Egypt Cairo|Canada Ottawa|Greece Athens|England London|Ireland Dublin
Austria Vienna|Poland Warsaw|Sweden Stockholm|Norway Oslo|Finland Helsinki|Denmark Copenhagen
Portugal Lisbon|Turkey Ankara|Iran Tehran|Iraq Baghdad|Syria Damascus|Kenya Nairobi|Peru Lima
Chile Santiago|Cuba Havana|Thailand Bangkok|Vietnam Hanoi|Korea Seoul|Israel Jerusalem
Lebanon Beirut|Hungary Budapest|Belgium Brussels|Netherlands Amsterdam|Australia Canberra
Argentina Buenos|Afghanistan Kabul|Pakistan Islamabad|Ukraine Kiev|Nigeria Abuja
Philippines Manila|Indonesia Jakarta|Venezuela Caracas|Colombia Bogota|Scotland Edinburgh
Wales Cardiff|Jordan Amman|Qatar Doha|Morocco Rabat|Sudan Khartoum|Libya Tripoli
Romania Bucharest|Serbia Belgrade|Bulgaria Sofia|Switzerland Bern|Mongolia Ulaanbaatar"""


def token(word: str) -> int | None:
    """The single GPT-2 token for `" " + word`, or None when the word splits."""
    ids = tokenizer.encode(" " + word)
    return ids[0] if len(ids) == 1 else None


pairs = []
for line in CAPITALS.replace("\n", "|").split("|"):
    country, capital = line.split()
    if token(country) is not None and token(capital) is not None:
        pairs.append((token(country), token(capital)))
rng = random.Random(0)
rng.shuffle(pairs)
test_pairs, example_pairs = pairs[: len(pairs) // 2], pairs[len(pairs) // 2 :]
Q, A, NEWLINE, BLANK = (tokenizer.encode(s) for s in ("Q:", "A:", "\n", "\n\n"))


def prompt(shots: list[tuple[int, int]], query: int) -> list[int]:
    """`Q: x\nA: y\n\n` per example, then `Q: query\nA:`; every prompt has the same length."""
    ids = [t for x, y in shots for t in (*Q, x, *NEWLINE, *A, y, *BLANK)]
    return ids + [*Q, query, *NEWLINE, *A]


RESULT = len(test_pairs) >= 20 and len(example_pairs) >= 20

## 1. In-context prompts, clean and with shuffled answers

Each clean prompt shows five country-capital examples and asks for one more. Countries whose
name or capital is not a single GPT-2 token are dropped. Its corrupted
twin keeps the same countries and capitals but shuffles which capital follows which
country, so the examples no longer demonstrate the task. Both have the same tokens per
position, which lets one head's output move between them.

In [ ]:
# recipe-test: tool=transformers expected=True
clean, shuffled, answers = [], [], []
for _ in range(10):
    *shots, (query, answer) = rng.sample(example_pairs, 6)
    capitals = [y for _, y in shots]
    deranged = capitals[:]
    while any(a == b for a, b in zip(deranged, capitals)):
        rng.shuffle(deranged)
    clean.append(prompt(shots, query))
    shuffled.append(prompt([(x, y) for (x, _), y in zip(shots, deranged)], query))
    answers.append(answer)
clean, shuffled, answers = torch.tensor(clean), torch.tensor(shuffled), torch.tensor(answers)
rows = torch.arange(len(answers))


def answer_logprob(logits: torch.Tensor) -> float:
    """Mean log-probability of the correct capital at the last position."""
    return logits[:, -1].log_softmax(-1)[rows, answers].mean().item()


with torch.no_grad():
    clean_lp = answer_logprob(model(clean).logits)
    shuffled_lp = answer_logprob(model(shuffled).logits)
print(f"answer log-prob: clean {clean_lp:.2f}, shuffled {shuffled_lp:.2f}")
RESULT = clean_lp > shuffled_lp

## 2. Per-head outputs at the last token

`attention_head_contributions` splits each attention layer's output into one row per head:
the head's own write to the residual stream (its value-weighted mix projected through its
slice of the output matrix). The rows are checked to sum to the layer's output, so nothing is
lost in the split. One trace of each prompt set is enough.

In [ ]:
# recipe-test: tool=transformers expected=144
last = [clean.shape[1] - 1]
clean_heads = mi.attention_head_contributions(tl.trace(model, clean, save="all"), positions=last)
shuffled_heads = mi.attention_head_contributions(
    tl.trace(model, shuffled, save="all"), positions=last
)
head_mean, head_shuffled, head_site = {}, {}, {}
for row, other in zip(clean_heads.rows, shuffled_heads.rows):
    label = row.coordinate.label  # e.g. "transformer.h.9.attn.head8"
    if ".head" in label:
        head_mean[label] = row.value.reshape(len(answers), -1).mean(0)
        head_shuffled[label] = other.value.reshape(len(answers), -1)
        head_site[label] = row.coordinate.module_address + ".c_proj"
RESULT = len(head_mean)

## 3. Each head's causal indirect effect

For one head, a bound hook at its layer's output projection adds, at the last position,
the difference between the head's mean clean output and its output on the shuffled prompt.
`tl.when(...).bind(model)` runs the model with that hook and no capture. The effect is how
far the correct capital's log-probability recovers. Only the last position changes, so the
shuffled prompts' first tokens run once and their key-value cache is reused: each of the
144 heads then costs a one-token forward.

In [ ]:
# recipe-test: tool=transformers expected=True
with torch.no_grad():
    prefix_cache = model(shuffled[:, :-1], use_cache=True).past_key_values


def head_effect(label: str) -> float:
    delta = head_mean[label] - head_shuffled[label]

    def add_mean(activation: torch.Tensor, *, ctx: object) -> torch.Tensor:
        activation = activation.clone()
        activation[:, -1] += delta
        return activation

    bound = tl.when(tl.module(head_site[label]), add_mean).bind(model)
    with torch.no_grad():
        logits = bound(
            shuffled[:, -1:], past_key_values=copy.deepcopy(prefix_cache), use_cache=True
        ).logits
    return answer_logprob(logits) - shuffled_lp


effect = {label: head_effect(label) for label in head_mean}
top_heads = sorted(effect, key=effect.get, reverse=True)[:10]
for label in top_heads:
    print(f"{label}: {effect[label]:+.3f}")
RESULT = effect[top_heads[0]] > 0.1

## 4. The function vector, added zero-shot

The function vector is the sum of the top heads' mean outputs. The test prompts are
`Q: <country>\nA:` for countries never used in an example, with no examples at all; GPT-2
small mostly repeats the country. The vector goes in with `tl.steer` at one layer's
attention output (so into the residual stream), at the last position only (`tl.positions`).
A random vector of the same norm is the control. The sweep over layers shows where the
model can use it.

In [ ]:
# recipe-test: tool=transformers expected=True
function_vector = sum(head_mean[label] for label in top_heads)
control = torch.randn(768, generator=torch.Generator().manual_seed(0))
control = control / control.norm() * function_vector.norm()
queries = torch.tensor([[*Q, x, *NEWLINE, *A] for x, _ in test_pairs])
targets = torch.tensor([y for _, y in test_pairs])


def steered(vector: torch.Tensor, layer: int) -> object:
    """The model with ``vector`` added at the last position of layer ``layer``'s attention."""
    edit = tl.positions(tl.steer(vector, 1.0, dim=-1), "last", assume_no_padding=True)
    return tl.when(tl.module(f"transformer.h.{layer}.attn.c_proj"), edit).bind(model)


def accuracy(run: object) -> float:
    with torch.no_grad():
        prediction = run(queries).logits[:, -1].argmax(-1)
    return (prediction == targets).float().mean().item()


baseline = accuracy(model)
by_layer = {
    layer: (accuracy(steered(function_vector, layer)), accuracy(steered(control, layer)))
    for layer in range(12)
}
print(f"no vector: {baseline:.2f}")
for layer, (fv_accuracy, control_accuracy) in by_layer.items():
    print(
        f"layer {layer:>2}: function vector {fv_accuracy:.2f}, random vector {control_accuracy:.2f}"
    )
best = max(by_layer, key=lambda layer: by_layer[layer][0])
RESULT = by_layer[best][0] > 0.5 + max(baseline, *(c for _, c in by_layer.values()))

A few answers at the best layer, without and with the vector:

In [ ]:
# recipe-test: tool=transformers expected=True
with torch.no_grad():
    plain = model(queries[:8]).logits[:, -1].argmax(-1)
    lifted = steered(function_vector, best)(queries[:8]).logits[:, -1].argmax(-1)
for (country, _), before, after in zip(test_pairs, plain.tolist(), lifted.tolist()):
    print(
        f"{tokenizer.decode([country]):>12}: {tokenizer.decode([before])!r} -> {tokenizer.decode([after])!r}"
    )
RESULT = (lifted == targets[:8]).sum().item() > (plain == targets[:8]).sum().item()

In the paper the same vector, taken from a few heads in the middle of a much larger model,
also transfers to natural-text templates and composes with other function vectors. To run
the recipe on another task, replace the word pairs; on another causal language model,
replace the site addresses (`transformer.h.{layer}.attn.c_proj`) with that model's
attention output projection.